# ETL Pipeline — ChurnCube
**Steps:** Load → Fix TotalCharges → Save raw → Encode categoricals → Save staging → Normalize numerics → Save curated

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

RAW_CSV = Path('../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

## 1. Load & Fix TotalCharges

In [2]:
df = pd.read_csv(RAW_CSV)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')

# TotalCharges is stored as string; blanks are 11 new customers (tenure=0)
df['TotalCharges'] = df['TotalCharges'].str.strip()
blank_mask = df['TotalCharges'] == ''
print(f'Blank TotalCharges: {blank_mask.sum()} rows')
print(df[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Impute: new customers (tenure=0) logically have TotalCharges = MonthlyCharges * 1 (first month)
# Safe fallback: impute with MonthlyCharges since tenure=0 means they just joined
df.loc[blank_mask, 'TotalCharges'] = df.loc[blank_mask, 'MonthlyCharges']
print(f'\nRemaining nulls in TotalCharges: {df["TotalCharges"].isna().sum()}')

Loaded: 7,043 rows × 21 cols
Blank TotalCharges: 11 rows
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             
3331  7644-OMVMY       0           19.85             
3826  3213-VVOLG       0           25.35             
4380  2520-SGTTA       0           20.00             
5218  2923-ARZLG       0           19.70             
6670  4075-WKNIU       0           73.35             
6754  2775-SEFEE       0           61.90             

Remaining nulls in TotalCharges: 0


In [3]:
# Save raw (only TotalCharges fixed, everything else original)
df.to_parquet(DATA_DIR / 'raw.parquet', index=False)
print('Saved → data/raw.parquet')
print(df.dtypes)

Saved → data/raw.parquet
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                   str
dtype: object


## 2. Encode Categoricals → Staging

In [4]:
df_staging = df.copy()

# --- Binary columns: map to 0/1 ---
binary_map = {'Yes': 1, 'No': 0, 'Female': 0, 'Male': 1}
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_staging[col] = df_staging[col].map(binary_map)
print('Binary encoded:', binary_cols)

# --- Service add-on columns: 3 values but 'No internet service' / 'No phone service' → both treated as 0 ---
# Rationale: customer doesn't have the add-on regardless of reason
addon_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
              'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
addon_map = {'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0}
for col in addon_cols:
    df_staging[col] = df_staging[col].map(addon_map)
print('Add-on service encoded:', addon_cols)

# --- Multi-category columns: one-hot encode ---
onehot_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_staging = pd.get_dummies(df_staging, columns=onehot_cols, drop_first=False)
# Clean up column names
df_staging.columns = [c.replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '') for c in df_staging.columns]
print('One-hot encoded:', onehot_cols)

print(f'\nStaging shape: {df_staging.shape}')
print(df_staging.dtypes)

Binary encoded: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
Add-on service encoded: ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
One-hot encoded: ['InternetService', 'Contract', 'PaymentMethod']

Staging shape: (7043, 28)
customerID                                   str
gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
MultipleLines                              int64
OnlineSecurity                             int64
OnlineBackup                               int64
DeviceProtection                           int64
TechSupport                                int64
StreamingTV                                int64
StreamingMovies             

In [5]:
df_staging.to_parquet(DATA_DIR / 'staging.parquet', index=False)
print('Saved → data/staging.parquet')

Saved → data/staging.parquet


## 3. Normalize Numerics → Curated (model-ready)

In [6]:
df_curated = df_staging.copy()

# Drop customerID — not a feature
df_curated = df_curated.drop(columns=['customerID'])

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_curated[numeric_cols] = scaler.fit_transform(df_curated[numeric_cols])

print('Scaled columns:', numeric_cols)
print(df_curated[numeric_cols].describe().round(3))
print(f'\nCurated shape: {df_curated.shape}')

Scaled columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
         tenure  MonthlyCharges  TotalCharges
count  7043.000        7043.000      7043.000
mean     -0.000          -0.000        -0.000
std       1.000           1.000         1.000
min      -1.318          -1.546        -0.998
25%      -0.952          -0.973        -0.830
50%      -0.137           0.186        -0.391
75%       0.921           0.834         0.665
max       1.614           1.794         2.826

Curated shape: (7043, 27)


In [7]:
df_curated.to_parquet(DATA_DIR / 'curated.parquet', index=False)
print('Saved → data/curated.parquet')
print('\nFinal schema:')
print(df_curated.dtypes)

Saved → data/curated.parquet

Final schema:
gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                   float64
PhoneService                               int64
MultipleLines                              int64
OnlineSecurity                             int64
OnlineBackup                               int64
DeviceProtection                           int64
TechSupport                                int64
StreamingTV                                int64
StreamingMovies                            int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
InternetService_DSL                         bool
InternetService_Fiber_optic                 bool
InternetService_No       

## 4. ETL Summary

In [8]:
print('=== ETL PIPELINE SUMMARY ===')
print(f'Raw CSV      : {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Staging      : {df_staging.shape[0]:,} rows × {df_staging.shape[1]} cols')
print(f'Curated      : {df_curated.shape[0]:,} rows × {df_curated.shape[1]} cols')
print()
print('Imputation   : 11 blank TotalCharges → MonthlyCharges value (all tenure=0)')
print('Binary cols  :', binary_cols)
print('Add-on cols  :', addon_cols, '(No/No service → 0, Yes → 1)')
print('One-hot cols :', onehot_cols)
print('Scaled cols  :', numeric_cols, '(StandardScaler)')
print('Dropped      : customerID')
print()
print('Output files :')
print('  data/raw.parquet      — cleaned TotalCharges, original structure')
print('  data/staging.parquet  — all encoded, numeric unscaled, customerID retained')
print('  data/curated.parquet  — fully model-ready, no customerID')

=== ETL PIPELINE SUMMARY ===
Raw CSV      : 7,043 rows × 21 cols
Staging      : 7,043 rows × 28 cols
Curated      : 7,043 rows × 27 cols

Imputation   : 11 blank TotalCharges → MonthlyCharges value (all tenure=0)
Binary cols  : ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
Add-on cols  : ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'] (No/No service → 0, Yes → 1)
One-hot cols : ['InternetService', 'Contract', 'PaymentMethod']
Scaled cols  : ['tenure', 'MonthlyCharges', 'TotalCharges'] (StandardScaler)
Dropped      : customerID

Output files :
  data/raw.parquet      — cleaned TotalCharges, original structure
  data/staging.parquet  — all encoded, numeric unscaled, customerID retained
  data/curated.parquet  — fully model-ready, no customerID
